In [7]:
!pip install wolframclient
!pip install cvxpy
from wolframclient.evaluation import WolframLanguageSession
from wolframclient.language import wl, wlexpr
import numpy as np
import cvxpy as cp


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [8]:

sentiments = {
    'Obsidian': '---',
    'Pyroflex': '---',
    'Thermalite': '++++',
    'Ashes': '-',
    'cake': '---',
    'Magma': '++++',
    'Scoria': '++',
    'Volcanic': '++',
    'Sulfur': '+++'
}

returns = {
    '+': 0.05,
    '++': 0.15,
    '+++': 0.25,
    '++++': 0.35,
    '-': -0.05,
    '--': -0.1,
    '---': -0.4,
    '----': -0.6
}

products = list(sentiments.keys())
print("Products:", products)

Products: ['Obsidian', 'Pyroflex', 'Thermalite', 'Ashes', 'cake', 'Magma', 'Scoria', 'Volcanic', 'Sulfur']


In [9]:
rets = np.array([returns[sentiments[products[i-1]]] for i in range(1,10)])
pi = cp.Variable(9)
objective = cp.Minimize(100 * cp.sum_squares(pi) - 10000 * rets.T @ pi)
constraints = [cp.norm(pi, 1) <= 100]
prob = cp.Problem(objective, constraints)

prob.solve()
print('Optimal allocation without integer constraints:')
for i in range(9):
    print("Position in ", products[i], ': ', f"{pi.value[i]:,.2f}", '%', sep='')

Optimal allocation without integer constraints:
Position in Obsidian: -17.19%
Position in Pyroflex: -17.19%
Position in Thermalite: 14.69%
Position in Ashes: 0.00%
Position in cake: -17.19%
Position in Magma: 14.69%
Position in Scoria: 4.69%
Position in Volcanic: 4.69%
Position in Sulfur: 9.69%


In [10]:
#building blocks for the Mathematica command
s1 = ' + '.join(['('+str(returns[sentiments[products[i-1]]])+')*p'+str(i)+'*10000-100*(p'+str(i)+')^2' for i in range(1,10)])
s2 = ' + '.join(['Abs[p'+str(i)+']' for i in range(1,10)]) + '<=100,'
s3 = ', '.join(['Element[p'+str(i)+', Integers]' for i in range(1,10)])
s4 = ', '.join(['p'+str(i) for i in range(1,10)])


In [11]:
'NMaximize[{'+s1+','+s2+s3+'}, {'+s4+'}]'


'NMaximize[{(-0.4)*p1*10000-100*(p1)^2 + (-0.4)*p2*10000-100*(p2)^2 + (0.35)*p3*10000-100*(p3)^2 + (-0.05)*p4*10000-100*(p4)^2 + (-0.4)*p5*10000-100*(p5)^2 + (0.35)*p6*10000-100*(p6)^2 + (0.15)*p7*10000-100*(p7)^2 + (0.15)*p8*10000-100*(p8)^2 + (0.25)*p9*10000-100*(p9)^2,Abs[p1] + Abs[p2] + Abs[p3] + Abs[p4] + Abs[p5] + Abs[p6] + Abs[p7] + Abs[p8] + Abs[p9]<=100,Element[p1, Integers], Element[p2, Integers], Element[p3, Integers], Element[p4, Integers], Element[p5, Integers], Element[p6, Integers], Element[p7, Integers], Element[p8, Integers], Element[p9, Integers]}, {p1, p2, p3, p4, p5, p6, p7, p8, p9}]'

In [1]:
with WolframLanguageSession() as session:
    val_max, sol = session.evaluate(wlexpr('NMaximize[{'+s1+','+s2+s3+'}, {'+s4+'}]'))
print("Maximum profit achievable:", val_max)

NameError: name 'WolframLanguageSession' is not defined

In [16]:
print("Percentage of capital used: ", sum([abs(el[1]) for el in sol]), '%', sep='')

Percentage of capital used: 78%


In [17]:
for i in range(9):
    print("Position in ", products[i], ': ', sol[i][1], '%', sep='')

Position in Obsidian: -5%
Position in Pyroflex: -5%
Position in Thermalite: 20%
Position in Ashes: -3%
Position in cake: -5%
Position in Magma: 13%
Position in Scoria: 7%
Position in Volcanic: 8%
Position in Sulfur: 12%
